# 02 — Clean: Parse, Validate, and Spatial Join

Loads `data/raw/requests_{YEAR}.parquet`, applies cleaning, performs a point-in-polygon
spatial join to assign census tract GEOIDs, and saves to `data/interim/`.

**Key findings from data validation (sample_311_2025.json):**
- `MethodReceived` values: Phone (53%), System (27%), API (16%), Internal (3%), Mail/Email (<2%)
  - System = city-initiated proactive inspections (Rat Rubout, Graffiti, Dirty Street proactives)
  - Internal = staff-logged requests (pothole pickups, footway repairs)
  - Resident-initiated = Phone + API + Mail + Email (~70%)
- Coordinate coverage: 73% overall, but **~99% for non-ECC resident types**
  - ECC- prefix types (info requests, vehicle lookup) are 92-100% missing coordinates — intentional, no address
- `CloseDate` / `SRStatus` consistency: perfect — no mismatches
- Negative `days_to_close`: sub-second precision artifacts in same-day closures; floor to 0
- `LastActivity` reopen signal: NONE. Values are 'Service Response' (69%) and NULL (30%) only
- `DueDate`: 100% populated; standardized per SRType (e.g. pothole=1d, water leak=2d, vacant building=15d)
  - ECC and some proactive types have DueDate < CreatedDate (artifact) — exclude from on-time rate
- `Agency` field has trailing whitespace — strip in pipeline
- `Outcome` field has non-breaking space (U+00A0) in some values — normalize

**Prerequisites:**
- `data/raw/requests_{YEAR}.parquet` (from `01_ingest.ipynb`)
- `data/raw/baltimore_tracts.geojson` — see cell below for one-time download

**Output:** `data/interim/requests_{YEAR}_clean.parquet`

In [ ]:
import sys
from pathlib import Path

sys.path.insert(0, str(Path('.').resolve().parent / 'src'))

import numpy as np
import pandas as pd
import geopandas as gpd

from balt311.metrics import (
    parse_timestamps,
    clean_strings,
    flag_request_source,
    compute_days_to_close,
    compute_due_date_gap,
)

YEAR      = 2024
RAW_DIR   = Path('..') / 'data' / 'raw'
INTERIM   = Path('..') / 'data' / 'interim'
INTERIM.mkdir(exist_ok=True)

IN_FILE    = RAW_DIR / f'requests_{YEAR}.parquet'
TRACTS_GEO = RAW_DIR / 'baltimore_tracts.geojson'
OUT_FILE   = INTERIM / f'requests_{YEAR}_clean.parquet'

In [ ]:
df = pd.read_parquet(IN_FILE)
print(f'Loaded {len(df):,} rows')
print(df.dtypes)

## 1. Parse timestamps, clean strings, compute derived fields

In [ ]:
df = parse_timestamps(df)
df = clean_strings(df)          # strips Agency whitespace, normalizes non-breaking spaces
df = flag_request_source(df)    # adds is_resident (Phone/API/Mail/Email = True)
df = compute_days_to_close(df)  # floored at 0 for sub-second precision artifacts
df = compute_due_date_gap(df)   # adds due_date_gap_days and is_on_time

print(f'CreatedDate range: {df["CreatedDate"].min()} → {df["CreatedDate"].max()}')
print(f'\nis_resident breakdown:')
print(df.groupby(['is_resident', 'MethodReceived']).size().to_string())
print(f'\ndays_to_close (all records):')
print(df['days_to_close'].describe(percentiles=[.25,.5,.75,.9,.95,.99]))

## 2. Coordinate coverage

Expected: ~73% overall, ~99% after excluding ECC- prefix types.

In [ ]:
n = len(df)
valid_coords = df['Latitude'].notna() & df['Longitude'].notna() & (df['Latitude'] != 0)
non_ecc = ~df['SRType'].str.startswith('ECC-', na=False)

print(f'All records:       {valid_coords.sum():,}/{n:,} have coords ({100*valid_coords.mean():.1f}%)')
print(f'Non-ECC records:   {(valid_coords & non_ecc).sum():,}/{non_ecc.sum():,} ({100*(valid_coords & non_ecc).sum()/non_ecc.sum():.1f}%)')
print(f'Resident non-ECC:  ', end='')
res_non_ecc = df['is_resident'] & non_ecc
print(f'{(valid_coords & res_non_ecc).sum():,}/{res_non_ecc.sum():,} ({100*(valid_coords & res_non_ecc).sum()/res_non_ecc.sum():.1f}%)')

df_geo = df[valid_coords].copy()
print(f'\nProceeding with {len(df_geo):,} geocoded records for spatial join.')

## 3. Spatial join to census tracts

One-time download of Baltimore City TIGER/Line tract boundaries (no key required):
```python
import urllib.request
url = ('https://tigerweb.geo.census.gov/arcgis/rest/services/TIGERweb/tigerWMS_Current'
       '/MapServer/8/query?where=STATE%3D24+AND+COUNTY%3D510&outFields=GEOID,NAME&f=geojson')
urllib.request.urlretrieve(url, '../data/raw/baltimore_tracts.geojson')
```

In [ ]:
if not TRACTS_GEO.exists():
    raise FileNotFoundError(
        f'{TRACTS_GEO} not found. See markdown cell above for download command.'
    )

tracts = gpd.read_file(TRACTS_GEO).to_crs('EPSG:4326')
print(f'Tracts: {len(tracts)} polygons')
print(tracts[['GEOID','NAME']].head())

In [ ]:
gdf = gpd.GeoDataFrame(
    df_geo,
    geometry=gpd.points_from_xy(df_geo['Longitude'], df_geo['Latitude']),
    crs='EPSG:4326',
)

joined = gpd.sjoin(gdf, tracts[['GEOID', 'geometry']], how='left', predicate='within')
joined = joined.rename(columns={'GEOID': 'tract_geoid'})

no_tract = joined['tract_geoid'].isna().sum()
print(f'Joined: {len(joined):,} rows')
print(f'No tract match: {no_tract:,} ({100*no_tract/len(joined):.1f}%)')

## 4. Save

In [ ]:
out = pd.DataFrame(joined.drop(columns=['geometry', 'index_right'], errors='ignore'))
out.to_parquet(OUT_FILE, index=False)
print(f'Saved {len(out):,} rows → {OUT_FILE}')
print(f'Columns: {list(out.columns)}')